<a href="https://colab.research.google.com/github/hardtmad/distance-mapping/blob/main/Distances.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Connect to Google Drive

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Read postal codes CSV

import pandas as pd

# Country codes: https://pgeocode.readthedocs.io/en/latest/
# Path to postcal codes file in Drive.
postal_codes_path = '/content/gdrive/MyDrive/Academics/Distances/country_postal_final.csv'

# Make sure we've cleared out the df.
df = pd.read_csv(postal_codes_path)
print(f"Number of rows: {len(df)}")
if not df.empty:
  del df
df = pd.read_csv(postal_codes_path)
# Check the df.
# print(df.head())

# Get basic descriptive statistics
desc_stats = df.describe()
print("Basic Descriptive Statistics:")
print(desc_stats)

In [ ]:
# Set up shapefiles.
import geopandas as gpd

# USA shapefile
# source census.gov
usa_shapefile_path = '/content/gdrive/MyDrive/Academics/Distances/tl_2025_us_state.shx'
usa = gpd.read_file(usa_shapefile_path)

# world shapefile for EU
# source https://www.naturalearthdata.com/
world_countries_shapefile_path = '/content/gdrive/MyDrive/Academics/Distances/ne_110m_admin_0_countries.shx'
eu = gpd.read_file(world_countries_shapefile_path)

In [ ]:
%pip install pgeocode

In [ ]:
# Convert postal codes to latitude, longitude.

import pgeocode
import pandas as pd

# Create an empty list to store geocoded DataFrames
geocoded_dfs = []

# Group by country and process each group
for country_code, country_df in df.groupby('country_code'):
    # Initialize Nominatim for the current country
    nomi = pgeocode.Nominatim(country_code.lower())

    # Query multiple postal codes for the current country group
    # Ensure the 'postal_code' column is treated as strings
    postal_codes_to_query = country_df['postal_code'].astype(str).tolist()
    df_locations = nomi.query_postal_code(postal_codes_to_query)

    # Merge the results back to include the original country and postal_code for context
    if not df_locations.empty:
        df_locations = df_locations.rename(columns={'postal_code': 'postal_code'}, errors='ignore')

        # Select only relevant columns from geocoded results and work on a copy to avoid SettingWithCopyWarning
        geocoded_country_data = df_locations[['postal_code', 'latitude', 'longitude']].copy()

        # Add country code back to the geocoded data for consistency
        geocoded_country_data['country_code'] = country_code
        geocoded_dfs.append(geocoded_country_data)

# Concatenate all geocoded DataFrames into a single DataFrame
if geocoded_dfs:
    geocoded_results_df = pd.concat(geocoded_dfs, ignore_index=True)
    print(f"Number of rows: {len(geocoded_results_df)}")
    # print("\nGeocoded Results:")
    print(geocoded_results_df.head())
else:
    print("No postal codes were geocoded.")

In [ ]:
# Plot points on USA

# Group nearby points by rounding their latitude and longitude coordinates. Plot
# locations with size of each market proportional to count of points it represents.

import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point

# Create a GeoDataFrame from points
geometry = [Point(xy) for xy in zip(geocoded_results_df['longitude'], geocoded_results_df['latitude'])]
gdf_points = gpd.GeoDataFrame(geocoded_results_df, geometry=geometry, crs="EPSG:4326")

# Plotting
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the shapefile (base map)
usa.plot(ax=ax, color='white', edgecolor='gray')

# Filter out points not in the USA
gdf_points_usa = gdf_points[gdf_points.within(usa.unary_union)]

# Define a function to round coordinates for grouping
def round_coords(gdf, decimal_places=1):
    gdf_copy = gdf.copy() # Work on a copy to avoid SettingWithCopyWarning
    gdf_copy['rounded_latitude'] = gdf_copy['latitude'].round(decimal_places)
    gdf_copy['rounded_longitude'] = gdf_copy['longitude'].round(decimal_places)
    return gdf_copy

# Apply rounding to the USA points
gdf_points_usa_rounded = round_coords(gdf_points_usa, decimal_places=1)

# Group by rounded coordinates and count points
grouped_usa_points = gdf_points_usa_rounded.groupby(['rounded_latitude', 'rounded_longitude']).size().reset_index(name='count')

# Convert grouped points back to a GeoDataFrame
# Use the rounded coordinates as the new 'geometry' for plotting the aggregated points
geometry_grouped_usa = [Point(xy) for xy in zip(grouped_usa_points['rounded_longitude'], grouped_usa_points['rounded_latitude'])]
gdf_grouped_usa = gpd.GeoDataFrame(grouped_usa_points, geometry=geometry_grouped_usa, crs="EPSG:4326")

# Plot the shapefile (base map)
usa.plot(ax=ax, color='white', edgecolor='gray')

# Plot the grouped points with size proportional to count
# Scale marker size for better visualization (adjust the multiplier as needed)
gdf_grouped_usa.plot(ax=ax, marker='o', color='#f2ab8f', markersize=gdf_grouped_usa['count'] * 20 + 20, label='People', zorder=2)

# Plot the wedding location using ax.text for a solid heart
lon_wedding = np.array([-104.99367290886074])  # Wedding longitude
lat_wedding = np.array([39.74176619456535])    # Wedding latitude
ax.text(lon_wedding[0], lat_wedding[0], '\u2665', fontsize=15, color='#173f41', ha='center', va='center', zorder=5)

# Create a plot entry for the legend (using an outlined heart for compatibility)
ax.plot([], [], linestyle='None', marker='$\heartsuit$', color='#173f41', markersize=10, label='Wedding')

# Add title and legend
# ax.set_title('Wedding Distances')
ax.legend(loc="lower left")

ax.set_xlim([-130, -65])
ax.set_ylim([24, 50])

# Save a high res map.
plt.savefig("high_res_usa.png", dpi=600, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
from shapely.geometry import Point
import pandas as pd
import geopandas as gpd

# Create a GeoDataFrame from points
geometry = [Point(xy) for xy in zip(geocoded_results_df['longitude'], geocoded_results_df['latitude'])]
gdf_points = gpd.GeoDataFrame(geocoded_results_df, geometry=geometry, crs="EPSG:4326")

# Filter out points in USA. All others are in Europe in this case.
gdf_points_eu = gdf_points[~gdf_points.within(usa.unary_union)].copy()

# Manually add the missing Stirling point (FKH 2HU, GB). Not sure why this isn't appearing.
# Coordinates for Stirling area: ~56.1165, -3.9369
stirling_lat, stirling_lon = 56.1165, -3.9369
stirling_point = gpd.GeoDataFrame({
    'postal_code': ['FKH 2HU'],
    'latitude': [stirling_lat],
    'longitude': [stirling_lon],
    'country_code': ['gb'],
    'geometry': [Point(stirling_lon, stirling_lat)]
}, crs="EPSG:4326")

# Append to the EU dataset
gdf_points_eu = pd.concat([gdf_points_eu, stirling_point], ignore_index=True)

# Apply rounding to the EU points
gdf_points_eu_rounded = round_coords(gdf_points_eu, decimal_places=1)

# Group by rounded coordinates and count points
grouped_eu_points = gdf_points_eu_rounded.groupby(['rounded_latitude', 'rounded_longitude']).size().reset_index(name='count')

# Convert grouped points back to a GeoDataFrame
geometry_grouped_eu = [Point(xy) for xy in zip(grouped_eu_points['rounded_longitude'], grouped_eu_points['rounded_latitude'])]
gdf_grouped_eu = gpd.GeoDataFrame(grouped_eu_points, geometry=geometry_grouped_eu, crs="EPSG:4326")

# Plotting
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the shapefile (base map)
eu.plot(ax=ax, color='white', edgecolor='gray')

# Plot the grouped points with size proportional to count
gdf_grouped_eu.plot(ax=ax, marker='o', color='#f2ab8f', markersize=gdf_grouped_eu['count'] * 20 + 20, label='People', zorder=2)

# Zoom in on Europe/UK area
ax.set_xlim([-10, 17])
ax.set_ylim([45, 61])

# Save a high res map.
plt.savefig("high_res_eu.png", dpi=600, bbox_inches='tight')

plt.show()